# 🕷️ Scraping Ulasan Kafe WFC Google Maps

**Nama :** Fadhli Ilham Nafi'an Yuswono (23.18.001)
**Jurusan :** Teknik Informatika S-1 — Institut Teknologi Nasional Malang
**Penelitian :** Nafinder — Rekomendasi Kafe WFC Kota Malang dengan Multinomial Naive Bayes

**Metode :** Selenium
**Input :** `daftar_kafe_final.csv`
**Output :** `ulasan_wfc_final.csv`

Tahapan: setup Chrome headless → baca daftar kafe → buka tab Ulasan per kafe → scroll & baca kartu ulasan → checkpoint CSV per 100 ulasan → hitung kolom `valid`.

> **Catatan Colab:** runtime bisa putus kapan saja, jadi scraping berjalan satu sel panjang yang menulis ke file CSV secara berkala. Bila terputus, jalankan ulang sel yang sama — `progress_wfc_final.json` menandai kafe mana yang sudah selesai, jadi proses otomatis melanjutkan dari kafe berikutnya. Setelah seluruh daftar kafe selesai, baru unduh `ulasan_wfc_final.csv`.


## ⚙️ IMPORT LIBRARY & INSTALASI DEPENDENSI

In [ ]:
!pip install -q selenium pandas

import os, re, csv, json, time, random, sys
from urllib.parse import quote_plus
import pandas as pd
from google.colab import files
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

print("✅ Libraries imported successfully!")


## 🧰 SETUP CHROME HEADLESS

Colab tidak punya Chrome/GUI, jadi Chromium dipasang lewat `apt` dan dijalankan mode headless. Bila tab Ulasan tidak muncul sama sekali meski sudah headless, kemungkinan kena CAPTCHA — solusinya jalankan script lokal (bukan Colab) dengan profil Chrome yang sudah login Google, mengikuti `scrapping Gmaps Nafii.py` di komputer sendiri.


In [ ]:
# Pasang Chromium + chromedriver (sekali per session Colab)
!apt-get update -qq > /dev/null
!apt-get install -y chromium-browser chromium-chromedriver > /dev/null 2>&1

FILE_KAFE = "daftar_kafe_final.csv"
FILE_HASIL = "ulasan_wfc_final.csv"
FILE_PROGRESS = "progress_wfc_final.json"

TARGET_PER_KAFE = 2000   # batas atas per kafe; berhenti juga kalau data habis
MAKS_KEKOSONGAN = 8      # berapa kali scroll tanpa ulasan baru sebelum nyerah
JEDA_SCROLL = (1.0, 2.0)
JEDA_ANTAR_KAFE = (8.0, 16.0)

KOLOM = [
    "placeId", "nama_kafe", "teks_ulasan", "rating_ulasan",
    "tanggal_ulasan", "kafe_id", "valid",
]

def bikin_driver():
    o = Options()
    o.add_argument("--headless=new")
    o.add_argument("--no-sandbox")
    o.add_argument("--disable-dev-shm-usage")
    o.add_argument("--lang=id")
    o.add_argument("--accept-language=id-ID,id")
    o.add_argument("--log-level=3")
    o.add_argument("--disable-notifications")
    o.add_argument("--disable-blink-features=AutomationControlled")
    o.add_experimental_option("excludeSwitches", ["enable-automation"])
    o.add_experimental_option("useAutomationExtension", False)
    service = Service("/usr/bin/chromedriver")
    return webdriver.Chrome(service=service, options=o)

print("✅ Setup Chrome headless siap.")


## 📥 UPLOAD DAFTAR KAFE

In [ ]:
uploaded = files.upload()
nama = list(uploaded.keys())[0]
assert nama == FILE_KAFE, f"Upload {FILE_KAFE}, bukan {nama!r}"

daftar = pd.read_csv(FILE_KAFE, encoding="utf-8-sig")
print(f"✅ Daftar kafe termuat: {len(daftar)} baris")
display(daftar.head())


## 🔧 UTILITAS PENCARIAN & VERIFIKASI

In [ ]:
def url_pencarian(nama, place_id=""):
    """URL pencarian resmi Google Maps (selalu ter-encode), bukan URL /place/ manual."""
    url = ("https://www.google.com/maps/search/?api=1"
           f"&query={quote_plus(nama)}&hl=id")
    if place_id:
        url += f"&query_place_id={place_id}"
    return url


def place_id_dari_url(url: str) -> str:
    m = re.search(r"(ChIJ[A-Za-z0-9_\-]{10,})", url or "")
    return m.group(1) if m else ""


def deteksi_kebijakan(driver) -> bool:
    """True kalau kena CAPTCHA / halaman 'belum memenuhi syarat'."""
    u = (driver.current_url or "").lower()
    if "sorry" in u or "interstitial" in u:
        return True
    try:
        t = (driver.find_element(By.TAG_NAME, "body").text or "").lower()[:800]
    except Exception:
        return False
    return any(k in t for k in ("tidak memenuhi syarat", "belum memenuhi syarat",
                                "sistem mendeteksi aktivitas", "unusual traffic"))


def baca_alamat(driver) -> str:
    for sel in ('a[data-item-id="address"] .Io6YTe',
                'button[data-item-id="address"] .Io6YTe',
                'button[data-item-id="address"]',
                'div.Io6YTe[data-value]'):
        try:
            t = (driver.find_element(By.CSS_SELECTOR, sel).text or "").strip()
            if t:
                return t
        except Exception:
            continue
    try:
        for b in driver.find_elements(By.CSS_SELECTOR, "button.CsEn2e, a.CsEn2e"):
            aria = (b.get_attribute("aria-label") or "")
            if "alamat" in aria.lower() or "address" in aria.lower():
                t = aria.split(":", 1)[-1].strip()
                if t:
                    return t
    except Exception:
        pass
    return ""


def aman_teks(driver, css, fallback=""):
    try:
        el = driver.find_element(By.CSS_SELECTOR, css)
        return (el.text or el.get_attribute("aria-label") or "").strip()
    except Exception:
        return fallback


def di_kota_malang(alamat: str) -> bool:
    """Khusus KOTA Malang — Kabupaten Malang harus terbuang."""
    a = (alamat or "").lower()
    if not a:
        return False
    if "kabupaten malang" in a:
        return False
    return ("kota malang" in a) or ("malang city" in a)


def kafe_id_dari(nama: str, alamat: str) -> str:
    jalan = (alamat.split(",")[0] or "").strip()
    return f"{nama.strip()} | {jalan}"


def nama_cocok(nama_csv, h1):
    """Verifikasi ringan: saling mengandung, tanpa huruf besar/kecil, spasi, tanda baca."""
    if not nama_csv or not h1:
        return False
    a = re.sub(r"[^a-z0-9]+", "", nama_csv.lower())
    b = re.sub(r"[^a-z0-9]+", "", h1.lower())
    return a in b or b in a


def jumlah_ulasan(driver):
    try:
        return len(driver.find_elements(By.CSS_SELECTOR, 'div[data-review-id]'))
    except Exception:
        return 0


def url_bersih(url: str) -> str:
    """Buang query string; URL tanpa query menampilkan tab Ulasan yang benar."""
    return str(url).split("?")[0]


print("✅ Utilitas siap.")


## 🖱️ KLIK & SCROLL TAB ULASAN

Google Maps pakai React, jadi klik Selenium biasa sering tidak sampai ke handler-nya. Ada empat strategi klik berurutan, lalu scroll hanya panel ulasan (bukan sidebar) agar peta/riwayat pencarian tidak ikut bergeser.


In [ ]:
def klik(driver, el):
    from selenium.webdriver.common.action_chains import ActionChains
    from selenium.common.exceptions import (
        ElementClickInterceptedException, ElementNotInteractableException)

    def pindah():
        try:
            for t in driver.find_elements(By.CSS_SELECTOR, 'div[role="tablist"] button'):
                s = ((t.get_attribute("aria-label") or "") + " " + (t.text or "")).lower()
                if any(k in s for k in ("ulasan", "review")):
                    if (t.get_attribute("aria-selected") == "true"
                            or t.get_attribute("aria-pressed") == "true"
                            or "selected" in (t.get_attribute("class") or "").lower()):
                        return True
        except Exception:
            pass
        return jumlah_ulasan(driver) > 0

    # 1) scroll + native click
    try:
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
        time.sleep(0.4)
        el.click()
        time.sleep(1.2)
        if pindah():
            return "native"
    except (ElementClickInterceptedException, ElementNotInteractableException, WebDriverException):
        pass

    # 2) ActionChains berbasis koordinat
    try:
        ActionChains(driver).move_to_element(el).pause(0.3).click().perform()
        time.sleep(1.2)
        if pindah():
            return "actionchains"
    except Exception:
        pass

    # 3) MouseEvent bubbles=true (React event delegation)
    for opts in ({"bubbles": True, "cancelable": True},
                 {"bubbles": True, "cancelable": True, "composed": True}):
        try:
            driver.execute_script(
                "arguments[0].dispatchEvent(new MouseEvent('click', arguments[1]));",
                el, opts)
            time.sleep(1.2)
            if pindah():
                return "mouseevent"
        except Exception:
            continue

    # 4) script click cadangan
    try:
        driver.execute_script("arguments[0].click();", el)
        time.sleep(1.2)
        if pindah():
            return "script-click"
    except Exception:
        pass

    return "gagal"


def scroll_panel_ulasan(driver):
    """Scroll hanya panel ulasan, bukan sidebar utama (agar peta tidak bergeser)."""
    jalur = driver.execute_script(r"""
      const rev = document.querySelector('div[data-review-id]');
      if (rev) {
        let el = rev.parentElement;
        while (el && el !== document.body) {
          if (el.scrollHeight > el.clientHeight + 40) {
            el.scrollTop = el.scrollHeight; return 'kartu';
          }
          el = el.parentElement;
        }
      }
      for (const e of document.querySelectorAll('div[class*="DxyBCb"]')) {
        if (e.scrollHeight > e.clientHeight + 40) {
          e.scrollTop = e.scrollHeight; return 'panel-tab';
        }
      }
      for (const b of document.querySelectorAll('button[aria-label]')) {
        const a = b.getAttribute('aria-label') || '';
        if (!/Urutkan ulasan|Saring ulasan|Telusuri ulasan/i.test(a)) continue;
        let p = b.parentElement;
        while (p && p !== document.body) {
          if (p.scrollHeight > p.clientHeight + 40) {
            p.scrollTop = p.scrollHeight; return 'dari-filter';
          }
          p = p.parentElement;
        }
      }
      return 'tidak-ada';
    """)
    if jalur != "tidak-ada":
        time.sleep(random.uniform(*JEDA_SCROLL))
    else:
        time.sleep(1)
    return jalur


def tab_ulasan_aktif(driver):
    try:
        for t in driver.find_elements(By.CSS_SELECTOR, 'div[role="tablist"] button'):
            s = ((t.get_attribute("aria-label") or "") + " " + (t.text or "")).lower()
            if "ulasan" in s or "review" in s:
                if (t.get_attribute("aria-selected") == "true"
                        or t.get_attribute("aria-pressed") == "true"
                        or "selected" in (t.get_attribute("class") or "").lower()):
                    return True
                return False
    except Exception:
        pass
    return None


def daftar_terbuka(driver, n, dasar=0):
    if n <= 0:
        return False
    if n > dasar:
        return True
    return tab_ulasan_aktif(driver) is not False


def tunggu_daftar_ulasan(driver, detik=20, dasar=0):
    batas = time.time() + detik
    while time.time() < batas:
        n = jumlah_ulasan(driver)
        if daftar_terbuka(driver, n, dasar):
            return n
        jalur = scroll_panel_ulasan(driver)
        if jalur == "tidak-ada":
            time.sleep(1)
    return jumlah_ulasan(driver)


print("✅ Strategi klik & scroll siap.")


## 📖 BACA KARTU ULASAN (SATU executeScript)

Satu pemanggilan JS membaca semua kartu sekaligus dan mengklik seluruh tombol "Lainnya" serentak — tanpa round-trip Selenium per kartu.


In [ ]:
def baca_ulasan(driver, expand=True):
    if expand:
        try:
            driver.execute_script("""
              for (const b of document.querySelectorAll('button.w8nwRe')) {
                try { b.click(); } catch (e) {}
              }
            """)
        except Exception:
            pass

    try:
        data = driver.execute_script(r"""
      const out = [];
      for (const c of document.querySelectorAll('div[data-review-id]')) {
        const rid = c.getAttribute('data-review-id') || '';
        const q = sel => {
          const e = c.querySelector(sel);
          return e ? e.textContent.trim() : '';
        };
        let rating = '';
        const r1 = c.querySelector('span.kvMYJc');
        const r2 = c.querySelector('div[role="img"][aria-label]');
        const lab = ((r1 && r1.getAttribute('aria-label')) ||
                     (r2 && r2.getAttribute('aria-label')) || '');
        const m = lab.match(/[1-5]/);
        if (m) rating = m[0];

        let teks = q('div.MyEned') || q('div.wiI7pd') || q('span[class*="MyEned"]');
        teks = teks.replace(/\s+/g, ' ').trim();

        let tgl_abs = '', tgl_rel = '';
        for (const s of c.querySelectorAll('span.rsqaWe, span[aria-label], span[title]')) {
          for (const raw of [s.getAttribute('aria-label'),
                             s.getAttribute('title'), s.textContent]) {
            if (!raw) continue;
            const t = raw.trim();
            if (!tgl_abs && /\b(19|20)\d{2}\b/.test(t)) tgl_abs = t;
            else if (!tgl_rel &&
                     /\b(lalu|hari|minggu|bulan|tahun|kemarin)\b/i.test(t)) tgl_rel = t;
          }
          if (tgl_abs) break;
        }
        out.push({id: rid, rating: rating, teks: teks,
                  tanggal_abs: tgl_abs, tanggal_rel: tgl_rel});
      }
      return out;
    """)
    except Exception as e:
        print(f"    ! baca DOM gagal ({type(e).__name__}), coba lagi dalam 3s")
        time.sleep(3)
        try:
            data = driver.execute_script(r"""
              const out = [];
              for (const c of document.querySelectorAll('div[data-review-id]')) {
                const q = sel => {
                  const e = c.querySelector(sel);
                  return e ? e.textContent.trim() : '';
                };
                const r = c.querySelector('span.kvMYJc')
                       || c.querySelector('div[role="img"][aria-label]');
                const m = ((r && r.getAttribute('aria-label')) || '').match(/[1-5]/);
                out.push({id: c.getAttribute('data-review-id') || '',
                          rating: m ? m[0] : '',
                          teks: (q('div.MyEned') || q('div.wiI7pd')).replace(/\s+/g,' ').trim(),
                          tanggal_abs: '', tanggal_rel: ''});
              }
              return out;
            """)
            return data or []
        except Exception:
            return []
    return data or []


print("✅ Pembaca kartu ulasan siap.")


## 🏠 BUKA KAFE & PINDAH KE TAB ULASAN

Tiga strategi berurutan: (A) klik tab berdasarkan teks "Ulasan", (B) klik tombol "N ulasan", (C) scroll panel ulasan saja. Kena CAPTCHA → return `None` (berhenti total); verifikasi nama gagal / di luar Kota Malang → return `False` (lewati kafe ini).


In [ ]:
def buka_tempat_dari_cari(driver, wait, nama, place_id=""):
    driver.get(url_pencarian(nama, place_id))
    time.sleep(random.uniform(3.0, 5.0))

    try:
        if driver.execute_script("return document.hidden;"):
            print("    ! PERINGATAN: tab Maps sedang di BELAKANG (bukan tab aktif).")
            print("      Render bisa tersendat -> kafe bisa terlewat gagal verifikasi.")
    except Exception:
        pass

    try:
        for _ in range(8):
            h1 = aman_teks(driver, "h1.DUwDvf") or aman_teks(driver, "h1")
            if h1 and "/place/" in (driver.current_url or ""):
                break
            hasil = driver.find_elements("css selector", "a.hfpxzc")
            if hasil:
                driver.execute_script("arguments[0].click();", hasil[0])
                time.sleep(4)
            else:
                time.sleep(1)
    except Exception:
        pass

    h1 = aman_teks(driver, "h1.DUwDvf") or aman_teks(driver, "h1")
    for _ in range(15):
        if h1:
            break
        time.sleep(1)
        h1 = aman_teks(driver, "h1.DUwDvf") or aman_teks(driver, "h1")
    return h1, driver.current_url


def simpan_debug(driver, nama):
    nm = re.sub(r"[^A-Za-z0-9]+", "_", (nama or "tanpa"))[:40]
    try:
        driver.save_screenshot(f"debug_{nm}.png")
        with open(f"debug_{nm}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"    debug tersimpan: debug_{nm}.png / debug_{nm}.html")
    except Exception as e:
        print(f"    debug gagal: {str(e)[:80]}")


def buka_tab_ulasan(driver, wait, url, nama_csv="", place_id=""):
    if place_id or nama_csv:
        h1_hal, url_akhir = buka_tempat_dari_cari(driver, wait, nama_csv or url, place_id)
        if not nama_cocok(nama_csv, h1_hal):
            print(f"    ! VERIFIKASI GAGAL: CSV={nama_csv!r} vs halaman={h1_hal!r}")
            print("      -> kafe dilewati (kemungkinan salah cabang)")
            simpan_debug(driver, nama_csv)
            return False
        print(f"    [cari] OK: {h1_hal!r} | {url_akhir[:90]}")
    else:
        driver.get(url_bersih(url))
        time.sleep(random.uniform(3.0, 5.0))

    if deteksi_kebijakan(driver):
        print("    !! KENAK CAPTCHA/KEBIJAKAN. Script berhenti total.")
        return None

    nama = aman_teks(driver, "h1.DUwDvf") or aman_teks(driver, "h1")
    alamat = baca_alamat(driver)

    if alamat and not di_kota_malang(alamat):
        print(f"    dilewati, di luar Kota Malang: {alamat[:70]}")
        return False

    dasar = jumlah_ulasan(driver)
    url_setelah_buka = driver.current_url

    def _dasar_place(u):
        u = (u or "").split("/data=")[0].split("?")[0]
        i = u.find("/@")
        return u[:i] if i > 0 else u

    def masih_di_tempat(nama_yg_dimaksud):
        h1 = aman_teks(driver, "h1.DUwDvf") or aman_teks(driver, "h1")
        if h1:
            return nama_cocok(nama_yg_dimaksud, h1)
        u = driver.current_url or ""
        return "/place/" in u and _dasar_place(u) == _dasar_place(url_setelah_buka)

    # Strategi A: klik tab berdasarkan teks
    try:
        tabs = driver.find_elements(By.CSS_SELECTOR, 'div[role="tablist"] button')
        idx = None
        for i, t in enumerate(tabs):
            s = ((t.get_attribute("aria-label") or "") + " " + (t.text or "")).lower()
            if any(k in s for k in ("ulasan", "review")):
                idx = i
                break
        if idx is not None:
            mode = klik(driver, tabs[idx])
            n = tunggu_daftar_ulasan(driver, detik=20, dasar=dasar)
            print(f"      [A] tab[{idx}] klik={mode} ulasan {dasar}->{n}")
            if daftar_terbuka(driver, n, dasar) and masih_di_tempat(nama):
                return (nama, alamat)
        elif len(tabs) >= 2:
            mode = klik(driver, tabs[1])
            n = tunggu_daftar_ulasan(driver, detik=20, dasar=dasar)
            print(f"      [A] tab[1] cadangan klik={mode} ulasan {dasar}->{n}")
            if daftar_terbuka(driver, n, dasar) and masih_di_tempat(nama):
                return (nama, alamat)
        else:
            print("      [A] tidak ada tab")
    except Exception as e:
        print(f"      [A] gagal: {str(e)[:90]}")

    # Strategi B: klik tombol "N ulasan"
    try:
        kandidat = driver.find_elements(
            By.CSS_SELECTOR,
            'div[role="tablist"] button, div[role="tablist"] [role="button"], '
            'div[role="main"] button')
        klik_dicoba = 0
        for el in kandidat:
            try:
                s = ((el.get_attribute("aria-label") or "") + " " + (el.text or "")).lower()
                if not re.search(r"\d\s*(ulasan|review)", s):
                    continue
                if not el.is_displayed():
                    continue
                klik_dicoba += 1
                klik(driver, el)
                n = tunggu_daftar_ulasan(driver, detik=15, dasar=dasar)
                if daftar_terbuka(driver, n, dasar) and masih_di_tempat(nama):
                    print(f"      [B] ok ulasan {dasar}->{n}")
                    alamat = alamat or baca_alamat(driver)
                    return (nama, alamat)
                if not masih_di_tempat(nama):
                    print("      [B] halaman pindah tempat -> buka ulang")
                    h1_hal, _ = buka_tempat_dari_cari(driver, wait, nama, place_id)
                    if not nama_cocok(nama, h1_hal):
                        print(f"      [B] buka ulang gagal ({h1_hal!r}) -> berhenti B")
                        break
                    dasar = jumlah_ulasan(driver)
                    tabs = driver.find_elements(By.CSS_SELECTOR, 'div[role="tablist"] button')
                    for t in tabs:
                        s2 = ((t.get_attribute("aria-label") or "") + " " + (t.text or "")).lower()
                        if "ulasan" in s2 or "review" in s2:
                            klik(driver, t)
                            tunggu_daftar_ulasan(driver, detik=15, dasar=dasar)
                            break
            except Exception:
                continue
        if klik_dicoba == 0:
            print("      [B] tidak ada tombol 'N ulasan' di tab/pesan tempat")
    except Exception as e:
        print(f"      [B] gagal: {str(e)[:90]}")

    # Strategi C: scroll panel ulasan saja
    jalur_scroll = scroll_panel_ulasan(driver)
    print(f"      [C] scroll awal: {jalur_scroll}")
    time.sleep(2)
    n = jumlah_ulasan(driver)
    if daftar_terbuka(driver, n, dasar) and masih_di_tempat(nama):
        print(f"      [C] ok setelah scroll {dasar}->{n}")
        alamat = alamat or baca_alamat(driver)
        return (nama, alamat)

    try:
        nm = re.sub(r"[^A-Za-z0-9]+", "_", (nama or "tanpa"))[:40]
        path = f"gagal_tab_{nm}.png"
        driver.save_screenshot(path)
        with open(f"gagal_tab_{nm}.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"    ! tab ulasan gagal (ulasan {dasar}->{jumlah_ulasan(driver)}) -> {path}")
    except Exception as e:
        print(f"    ! tab ulasan gagal, screenshot pun gagal: {str(e)[:70]}")
    return False


print("✅ Navigasi kafe & tab ulasan siap.")


## 📊 LOOP UTAMA SCRAPING

In [ ]:
def kumpul_ulasan(driver, nama_kafe, place_id, kafe_id,
                  target=TARGET_PER_KAFE, simpan_batch=None):
    """Scroll sampai `target` ulasan atau data berhenti bertambah."""
    kumpul, kekosongan = {}, 0
    detik_mulai = time.time()
    t_sabit = 0
    berikut_batch = 100

    while len(kumpul) < target and kekosongan < MAKS_KEKOSONGAN:
        sebelum = len(kumpul)
        for r in baca_ulasan(driver):
            if not r["teks"]:
                continue
            kunci = r["id"] or r["teks"][:80]
            if kunci not in kumpul:
                kumpul[kunci] = r

        if len(kumpul) == sebelum:
            kekosongan += 1
        else:
            kekosongan = 0

        lewat = int(time.time() - detik_mulai)
        if lewat - t_sabit >= 4 or len(kumpul) >= target:
            t_sabit = lewat
            print(f"    [jalan] {len(kumpul):>4}/{target} ulasan "
                  f"| kekosongan {kekosongan}/{MAKS_KEKOSONGAN} "
                  f"| {lewat}s", flush=True)

        if simpan_batch and len(kumpul) >= berikut_batch:
            simpan_batch(list(kumpul.values()))
            berikut_batch = (len(kumpul) // 100 + 1) * 100

        if len(kumpul) >= target:
            break
        scroll_panel_ulasan(driver)

    durasi = int(time.time() - detik_mulai)
    print(f"    -> {len(kumpul)} ulasan ({durasi}s)")
    return list(kumpul.values())


def muat_progress():
    if os.path.exists(FILE_PROGRESS):
        try:
            return json.load(open(FILE_PROGRESS, encoding="utf-8"))
        except Exception:
            return []
    return []


def simpan_progress(done):
    json.dump(done, open(FILE_PROGRESS, "w", encoding="utf-8"), ensure_ascii=False)


def teks_valid(teks: str, seen: set) -> bool:
    """1 kosong -> False; 2 < 3 kata -> False; 3 duplikat (keep=first) -> False."""
    t = (teks or "").strip()
    if not t:
        return False
    if len(t.split()) < 3:
        return False
    if t in seen:
        return False
    seen.add(t)
    return True


def hapus_baris_kafe(path, place_id):
    """Hapus baris parsial milik satu kafe sebelum kafe diproses ulang (hindari ganda)."""
    if not place_id or not os.path.exists(path) or os.path.getsize(path) == 0:
        return 0
    with open(path, newline="", encoding="utf-8-sig") as f:
        semua = list(csv.reader(f))
    if len(semua) < 2:
        return 0
    header, isi = semua[0], semua[1:]
    try:
        i_pid = header.index("placeId")
    except ValueError:
        return 0
    sisa = [r for r in isi if len(r) <= i_pid or r[i_pid] != place_id]
    buang = len(isi) - len(sisa)
    if buang:
        with open(path, "w", newline="", encoding="utf-8-sig") as f:
            w = csv.writer(f)
            w.writerow(header)
            w.writerows(sisa)
    return buang


print("✅ Fungsi loop & checkpoint siap.")


## 🚀 MULAI SCRAPING

Sel ini panjang — berjalan sampai seluruh daftar kafe selesai atau kena CAPTCHA. `progress_wfc_final.json` menyimpan daftar `placeId` yang sudah beres, sehingga bila dijalankan ulang setelah terputus, kafe yang sudah selesai otomatis dilewati.


In [ ]:
TARGET_KESELESAIAN = 2000  # batas ulasan per kafe, bisa diganti sesuai kebutuhan

driver = bikin_driver()
wait = WebDriverWait(driver, 20)
berhenti_total = False

done = set(muat_progress())
teks_sudah_pakai = set()

tulis_header = not os.path.exists(FILE_HASIL)
fcsv = open(FILE_HASIL, "a", newline="", encoding="utf-8-sig")
out = csv.DictWriter(fcsv, fieldnames=KOLOM)
if tulis_header:
    out.writeheader()

print(f"📋 Daftar kafe: {len(daftar)} | sudah selesai: {len(done)} | target/kafe: {TARGET_KESELESAIAN}")

try:
    for idx, kafe in daftar.iterrows():
        link = kafe.get("link") or kafe.get("url") or ""
        if not link:
            continue
        pid = kafe.get("place_id") or place_id_dari_url(link)
        if pid and pid in done:
            print(f"[{idx+1}/{len(daftar)}] dilewati (sudah): {kafe.get('nama')}")
            continue

        print(f"[{idx+1}/{len(daftar)}] {kafe.get('nama')}")

        try:
            hasil_buka = buka_tab_ulasan(
                driver, wait, link, nama_csv=kafe.get("nama", ""), place_id=pid)
        except WebDriverException as e:
            print(f"    ! error driver: {e}")
            break

        if hasil_buka is None:
            berhenti_total = True
            break
        if hasil_buka is False:
            continue

        nama, alamat = hasil_buka
        if not alamat:
            print("    ! alamat tidak terbaca — tetap diproses, cek manual di CSV")

        if jumlah_ulasan(driver) == 0:
            print("    ! ulasan sama sekali tidak muncul, dilewati")
            continue

        kid = kafe_id_dari(nama, alamat or "")

        fcsv.flush(); fcsv.close()
        buang = hapus_baris_kafe(FILE_HASIL, pid)
        fcsv = open(FILE_HASIL, "a", newline="", encoding="utf-8-sig")
        out = csv.DictWriter(fcsv, fieldnames=KOLOM)
        if buang:
            print(f"    [hapus] {buang} baris parsial lama dibuang (diproses ulang)")

        def buat_simpan(pid=pid, nama=nama, kid=kid):
            posisi_terakhir = [0]
            def simpan_batch(rows):
                baru = rows[posisi_terakhir[0]:]
                if not baru:
                    return
                for r in baru:
                    out.writerow({
                        "placeId": pid,
                        "nama_kafe": nama,
                        "teks_ulasan": r["teks"],
                        "rating_ulasan": r["rating"],
                        "tanggal_ulasan": r["tanggal_abs"] or r["tanggal_rel"],
                        "kafe_id": kid,
                        "valid": str(teks_valid(r["teks"], teks_sudah_pakai)),
                    })
                fcsv.flush()
                posisi_terakhir[0] = len(rows)
            return simpan_batch

        simpan = buat_simpan()
        rows = kumpul_ulasan(driver, nama, pid, kid,
                             target=TARGET_KESELESAIAN, simpan_batch=simpan)
        simpan(rows)
        fcsv.flush()

        if not rows:
            print("    ! 0 ulasan terbaca TERSIMPAN — TIDAK ditandai selesai")
            continue

        if pid:
            done.add(pid)
            simpan_progress(list(done))

        time.sleep(random.uniform(*JEDA_ANTAR_KAFE))

finally:
    try:
        driver.quit()
    except Exception:
        pass
    fcsv.close()

if berhenti_total:
    print("\n⚠️ Berhenti karena CAPTCHA. Tunggu beberapa jam, lalu jalankan ulang sel ini — kafe yang sudah selesai otomatis dilewati.")
else:
    print(f"\n✅ Selesai. Hasil -> {FILE_HASIL}")


## ✅ HITUNG ULANG KOLOM `valid`

In [ ]:
df_hasil = pd.read_csv(FILE_HASIL, encoding="utf-8-sig")
t = df_hasil["teks_ulasan"].fillna("").str.strip()

m_kosong = t == ""
sisa = df_hasil[~m_kosong]
m_pendek = sisa["teks_ulasan"].str.strip().str.split().str.len().lt(3)
sisa2 = sisa[~m_pendek]
m_duplikat = sisa2.duplicated(subset=["teks_ulasan"], keep="first")
valid_idx = sisa2[~m_duplikat].index

df_hasil["valid"] = df_hasil.index.isin(valid_idx)
df_hasil.to_csv(FILE_HASIL, index=False, encoding="utf-8-sig")

print("✅ FINALISASI SELESAI")
print(f"  Total awal        : {len(df_hasil)}")
print(f"  Kosong            : {int(m_kosong.sum())}")
print(f"  Kurang dari 3 kata: {int(m_pendek.sum())}")
print(f"  Duplikat          : {int(m_duplikat.sum())}")
print(f"  Data valid        : {int(df_hasil['valid'].sum())}")


## 💾 UNDUH HASIL AKHIR

In [ ]:
print(f"📊 Total baris: {len(df_hasil)} | valid: {int(df_hasil['valid'].sum())}")
print(f"✅ Mengunduh: {FILE_HASIL}")
files.download(FILE_HASIL)
